# M2 Joint N/V LightGCN 빠른 1차 실험

H&M 60일과 Dunnhumby 전체 기간을 순차로 실행합니다. 각 데이터에서 `M1` 대비 `joint_nv`만 빠르게 비교하며, seed 42 validation만 사용합니다. test/holdout은 만들지 않습니다.

각 epoch의 진행 상태와 재개 checkpoint는 Drive의 결과 폴더에 저장됩니다. 연결이 끊긴 뒤 같은 노트북을 처음부터 다시 실행하면 마지막 저장 epoch에서 이어서 학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = 'c9f9989133f54e92c1125659389839bcc9a12cc2'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)

In [ ]:
import json, torch
from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_joint_nv_run, preflight_summary, run_two_dataset_screening
)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
hm_cfg = configure_joint_nv_run('hm', short_hm=True)
dh_cfg = configure_joint_nv_run('dunnhumby', short_hm=False)
print('===== H&M 60일 실행 설정 =====')
print(json.dumps(preflight_summary(hm_cfg), ensure_ascii=False, indent=2))
print('===== Dunnhumby 전체 기간 실행 설정 =====')
print(json.dumps(preflight_summary(dh_cfg), ensure_ascii=False, indent=2))
print('review complete: 다음 셀을 실행하면 학습이 시작됩니다.')

In [ ]:
results = run_two_dataset_screening()

In [ ]:
for label, frame in results.items():
    print(f'\n===== {label}: M1 vs joint_nv =====')
    columns = [
        'model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50', 'revenue@10', 'arp@10', 'coverage@10',
        'n_distinct@10', 'exposure_entropy@10', 'eff_catalog@10',
        'top10_share@10', 'top100_share@10', 'value_alignment'
    ]
    display(frame[[c for c in columns if c in frame.columns]])
    print('판정:', frame.attrs['decision'])
    delta = pd.read_csv(frame.attrs['result_paths']['delta_csv'])
    display(delta)
    print('결과 파일:', frame.attrs['result_paths'])
print('완료. 위 표와 판정을 그대로 공유해 주세요.')